# Position Density Animation

Load and animate the persisted position-density snapshots for one observation session.

In [ ]:
# Load the shared project-path, database, and report-header helpers.
%run pathutils.ipynb
%run database.ipynb
%run report-header.ipynb

In [ ]:
# Select the observation session and the delay between animation frames.
session_id = 12
frame_interval_ms = 500

# Enable either format after previewing the animation; exports are written into data/reports.
export_mp4 = True
export_gif = True

In [ ]:
# Display the standard database and report-generation metadata.
report_metadata = display_report_header('Position Density Animation')

In [ ]:
# Load snapshot metadata and populated cells using the shared SQL-query convention.
if not isinstance(session_id, int) or session_id <= 0:
    raise ValueError('session_id must be a positive integer')

snapshot_query = construct_query(
    'tracker',
    'reports',
    'position-density-snapshots.sql',
    {'session_id': session_id})
snapshot_cells = query_data('tracker', snapshot_query)
snapshot_cells['Captured At UTC'] = pd.to_datetime(snapshot_cells['Captured At UTC'], utc=True)
snapshot_cells.head(20)

In [ ]:
# Present one row per available frame before constructing the animation.
snapshot_summary = (snapshot_cells[[
    'Snapshot Id', 'Session Id', 'Captured At UTC', 'Position Count', 'Maximum Bin Count'
]]
.drop_duplicates('Snapshot Id')
.sort_values(['Captured At UTC', 'Snapshot Id'])
.reset_index(drop=True))
snapshot_summary

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from matplotlib import animation, colors
from IPython.display import HTML
from pathlib import Path

In [ ]:
def padded_limits(minimum, maximum):
    """
    Return stable chart limits with the same four-percent padding used by the UI
    """
    # Preserve useful axes even for legacy snapshots whose bounds collapse to one coordinate.
    padding = max((maximum - minimum) * 0.04, 0.01)
    return minimum - padding, maximum + padding


In [ ]:
def draw_frame(frame_number):
    """
    Replace the hexagon collection and heading with one complete persisted snapshot
    """
    # Select the complete persisted cell collection belonging to this snapshot.
    metadata = frame_metadata[frame_number]
    cells = snapshot_cells[
        snapshot_cells['Snapshot Id'] == metadata['Snapshot Id']
    ].dropna(subset=['Cell Latitude', 'Cell Longitude', 'Cell Count'])

    # Matplotlib expects horizontal/vertical offsets, so keep longitude before latitude.
    offsets = cells[['Cell Longitude', 'Cell Latitude']].to_numpy()
    counts = cells['Cell Count'].to_numpy()
    scatter.set_offsets(offsets if len(offsets) else [[float('nan'), float('nan')]])

    # Match the UI's logarithmic, frame-relative scale so sparse structure remains visible.
    frame_maximum = max(1, int(metadata['Maximum Bin Count']))
    relative_density = (
        np.log(counts + 1) / np.log(frame_maximum + 1) if len(counts) else np.array([np.nan]))
    scatter.set_array(relative_density)
    scatter.set_facecolors(
        colour_map(colour_scale(relative_density)) if len(counts) else [[0, 0, 0, 0]])

    # Keep hexagons a constant size: colour, rather than area, communicates density in the UI.
    scatter.set_sizes(np.full(len(counts), 82) if len(counts) else [0])
    captured = metadata['Captured At UTC'].strftime('%Y-%m-%d %H:%M:%S UTC')
    axis.set_title(
        f"SESSION {session_id}  ·  SNAPSHOT {frame_number + 1}/{len(frame_metadata)}  ·  {captured}\n"
        f"{metadata['Position Count']:,} positions     {len(cells):,} occupied bins     "
        f"{frame_maximum:,} peak density",
        color=text, fontsize=12, fontweight=700, pad=17, linespacing=1.7)
    return scatter,

In [ ]:
def export_movie(position_density_animation, movie_fps=2, movie_dpi=120):
    """
    Export the animation as an MP4 file
    """
    if movie_fps <= 0:
        raise ValueError('movie_fps must be greater than zero')
    if movie_dpi <= 0:
        raise ValueError('movie_dpi must be greater than zero')
    if not animation.writers.is_available('ffmpeg'):
        raise RuntimeError(
            'FFmpeg is required for MP4 export but was not found on PATH. '
            'Install FFmpeg, restart the notebook kernel, and run the notebook again.')

    # Use the established reports output folder and a session-specific deterministic filename.
    movie_path = Path(get_export_folder_path()) / f'position-density-session-{session_id}.mp4'
    movie_writer = animation.FFMpegWriter(
        fps=movie_fps,
        codec='libx264',
        metadata={
            'title': f'Position Density · Session {session_id}',
            'artist': 'ADS-B Base Station Reader'
        },
        extra_args=['-pix_fmt', 'yuv420p'])

    position_density_animation.save(
        movie_path,
        writer=movie_writer,
        dpi=movie_dpi,
        savefig_kwargs={'facecolor': background})

    print(f'Exported {len(frame_metadata)} frames to {movie_path}')

In [ ]:
def export_animated_gif(position_density_animation, movie_fps=2, movie_dpi=120):
    """
    Export the animation as a GIF file using Pillow.
    """
    if movie_fps <= 0:
        raise ValueError('movie_fps must be greater than zero')
    if movie_dpi <= 0:
        raise ValueError('movie_dpi must be greater than zero')
    if not animation.writers.is_available('pillow'):
        raise RuntimeError(
            'Pillow is required for GIF export but is not available. '
            'Install the reports dependencies, restart the notebook kernel, and run the notebook again.')

    # GIF uses the same frame rate and resolution controls as MP4 for consistent output.
    gif_path = Path(get_export_folder_path()) / f'position-density-session-{session_id}.gif'
    gif_writer = animation.PillowWriter(
        fps=movie_fps,
        metadata={
            'title': f'Position Density · Session {session_id}',
            'artist': 'ADS-B Base Station Reader'
        })

    position_density_animation.save(
        gif_path,
        writer=gif_writer,
        dpi=movie_dpi,
        savefig_kwargs={'facecolor': background})

    print(f'Exported {len(frame_metadata)} frames to {gif_path}')

In [ ]:
# Convert the compact summary once so animation frames can access metadata without repeated DataFrame lookups.
frame_metadata = snapshot_summary.to_dict('records')

# Use the union of persisted bounds so the viewport remains fixed while frames advance.
minimum_latitude = snapshot_cells['Minimum Latitude'].min()
maximum_latitude = snapshot_cells['Maximum Latitude'].max()
minimum_longitude = snapshot_cells['Minimum Longitude'].min()
maximum_longitude = snapshot_cells['Maximum Longitude'].max()

# Reproduce the five discrete Viridis bands used by the integrated SVG chart.
ui_colours = ['#365c8d', '#277f8e', '#1fa187', '#73d055', '#fde725']
colour_map = colors.ListedColormap(ui_colours)
colour_scale = colors.BoundaryNorm([0, .2, .4, .6, .8, 1.000001], colour_map.N)

# Match the integrated UI's chart-card palette and typography as closely as Matplotlib permits.
background = '#07111f'
surface = '#0d1b2d'
line = '#213750'
grid = '#20354c'
axis_line = '#526a82'
text = '#eff6ff'
muted = '#8fa6be'
axis_text = '#b8cada'

figure, axis = plt.subplots(figsize=(11.5, 7.2), facecolor=background)
axis.set_facecolor(surface)
axis.set_xlim(*padded_limits(minimum_longitude, maximum_longitude))
axis.set_ylim(*padded_limits(minimum_latitude, maximum_latitude))
axis.set_xlabel('Longitude', color=axis_text, labelpad=12)
axis.set_ylabel('Latitude', color=axis_text, labelpad=12)
axis.tick_params(colors=muted, labelsize=9)
axis.locator_params(axis='both', nbins=5)
axis.grid(color=grid, linestyle=(0, (3, 5)), linewidth=0.9, alpha=0.9)
axis.set_axisbelow(True)
for spine in axis.spines.values():
    spine.set_color(axis_line)
    spine.set_linewidth(1.2)

# Seed the collection with an invisible value so Matplotlib retains its colormap during animation.
scatter = axis.scatter(
    [float('nan')], [float('nan')], c=[0], s=[0],
    marker='h', cmap=colour_map, norm=colour_scale,
    edgecolors=(184 / 255, 202 / 255, 218 / 255, .65), linewidths=.8)

# Present the relative five-band scale as the compact Fewer/More legend used by the UI.
colour_bar = figure.colorbar(
    scatter, ax=axis, orientation='horizontal', pad=.13, fraction=.055,
    ticks=[.1, .3, .5, .7, .9], boundaries=[0, .2, .4, .6, .8, 1])
colour_bar.ax.set_xticklabels(['Fewer', '', '', '', 'More observations'])
colour_bar.ax.tick_params(colors=muted, labelsize=9, length=0)
colour_bar.outline.set_edgecolor(line)
figure.subplots_adjust(left=.09, right=.97, top=.84, bottom=.19)

# Retain the animation object so Jupyter does not garbage-collect it before producing the HTML player.
position_density_animation = animation.FuncAnimation(
    figure,
    draw_frame,
    frames=len(frame_metadata),
    interval=frame_interval_ms,
    repeat=True,
    blit=False)

# Suppress a duplicate static Matplotlib figure and display only the interactive animation.
plt.close(figure)

In [ ]:
if export_mp4:
    export_movie(position_density_animation)

if export_gif:
    export_animated_gif(position_density_animation)

In [ ]:
HTML(position_density_animation.to_jshtml())